# Private Property (Condo/Apartment) Price Prediction
**FYP-26-S1-13 | PropAI.sg**

Stacked generalisation model (XGBoost + CatBoost + LightGBM → HuberRegressor metalayer)
trained on private residential transactions fetched live from the **URA Data Service API**.

### Prerequisites
1. Register for a free URA API access key at: https://www.ura.gov.sg/maps/api/reg.html
2. Set `URA_ACCESS_KEY` in the cell below.
3. Google Drive access for SORA data (`data/SORA_2017_2026.xlsx`).

## 1  Install & Import

In [ ]:
!pip install -q xgboost lightgbm catboost scikit-learn pandas numpy requests openpyxl

In [ ]:
import os, re, time, json, requests
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import HuberRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

## 2  URA API — Fetch Private Residential Transactions
The URA Data Service provides up to 12 quarters of private residential
property transactions split across 4 batches.

**Token is valid for 24 hours.** Re-run `get_ura_token()` if you get a 401.

In [ ]:
# ── Set your URA access key here ──────────────────────────────────────────
URA_ACCESS_KEY = 'YOUR_ACCESS_KEY_HERE'   # https://www.ura.gov.sg/maps/api/reg.html
# ──────────────────────────────────────────────────────────────────────────

URA_BASE = 'https://www.ura.gov.sg/uraDataService/invokeUraDS'

def get_ura_token(access_key):
    url = f'{URA_BASE}?service=Token&AccessKey={access_key}'
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()
    if data.get('Status') == 'Success':
        return data['Result']
    raise ValueError(f'URA token error: {data}')

def fetch_ura_batch(access_key, token, batch):
    """Fetch one batch (1-4) of private residential transactions."""
    url = f'{URA_BASE}?service=PMI_Resi_Transaction&batch={batch}'
    headers = {'AccessKey': access_key, 'Token': token}
    r = requests.get(url, headers=headers, timeout=120)
    r.raise_for_status()
    data = r.json()
    if data.get('Status') == 'Success':
        return data.get('Result', [])
    print(f'Batch {batch} error: {data.get("Message", "")}')
    return []

token = get_ura_token(URA_ACCESS_KEY)
print(f'Token obtained: {token[:20]}...')

In [ ]:
all_projects = []
for batch in range(1, 5):
    projects = fetch_ura_batch(URA_ACCESS_KEY, token, batch)
    all_projects.extend(projects)
    print(f'Batch {batch}: {len(projects):,} project records')
    time.sleep(1)   # respect rate limits
print(f'\nTotal project records: {len(all_projects):,}')

### 2.1  Flatten nested transaction structure
Each project record contains a list of transactions. We unpack them into rows.

In [ ]:
def parse_contract_date(d):
    """MMYY format (e.g. '0124' = Jan 2024) → (month_int, year_int)"""
    try:
        d = str(d).zfill(4)
        mm, yy = int(d[:2]), int(d[2:])
        year = 2000 + yy if yy <= 99 else yy
        return mm, year
    except:
        return None, None

def parse_floor_range(fr):
    """'06-10' → 8.0  |  'B1' → -1  |  None → None"""
    if not fr or not isinstance(fr, str):
        return None
    if fr.strip().upper() in ('B1','B2','B3'):
        return -1.0
    m = re.search(r'(\d+)\s*[-–]\s*(\d+)', fr)
    if m:
        return (int(m.group(1)) + int(m.group(2))) / 2
    m2 = re.search(r'(\d+)', fr)
    return float(m2.group(1)) if m2 else None

def parse_tenure(t):
    """Returns (tenure_type: str, remaining_years: float)"""
    if not t or not isinstance(t, str):
        return 'Unknown', 50.0
    tl = t.lower()
    if 'freehold' in tl or '999' in tl:
        return 'Freehold', 999.0
    m_yrs  = re.search(r'(\d+)\s*yr', tl)
    m_comm = re.search(r'from\s+(\d{4})', tl)
    if m_yrs and m_comm:
        total_yrs   = int(m_yrs.group(1))
        commence_yr = int(m_comm.group(1))
        remaining   = max(0, commence_yr + total_yrs - 2025)
        return 'Leasehold', float(remaining)
    if m_yrs:
        return 'Leasehold', float(m_yrs.group(1))
    return 'Leasehold', 50.0

rows = []
for proj in all_projects:
    district       = proj.get('district', '')
    market_seg     = proj.get('marketSegment', '')
    x_svy          = proj.get('x')
    y_svy          = proj.get('y')
    for txn in proj.get('transaction', []):
        mm, yr = parse_contract_date(txn.get('contractDate', ''))
        ten_type, rem_yrs = parse_tenure(txn.get('tenure', ''))
        rows.append({
            'district':          int(district) if district else None,
            'market_segment':    market_seg,
            'property_type':     txn.get('propertyType', ''),
            'type_of_sale':      txn.get('typeOfSale', ''),
            'tenure_type':       ten_type,
            'remaining_tenure_yrs': rem_yrs,
            'area_sqm':          float(txn.get('area', 0) or 0),
            'floor_mid':         parse_floor_range(txn.get('floorRange', '')),
            'no_of_units':       int(txn.get('noOfUnits', 1) or 1),
            'price':             float(txn.get('price', 0) or 0),
            'month_num':         mm,
            'year':              yr,
        })

df = pd.DataFrame(rows)
print(df.shape)
df.head(3)

## 3  Data Exploration

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
print('Market segments:', df['market_segment'].value_counts().to_dict())
print('Property types:', df['property_type'].value_counts().to_dict())
print('Sale types:', df['type_of_sale'].value_counts().to_dict())
print('Tenure types:', df['tenure_type'].value_counts().to_dict())

## 4  Feature Engineering

In [ ]:
# Time features
df['quarter']  = ((df['month_num'] - 1) // 3 + 1)
df['time_idx'] = (df['year'] - df['year'].min()) * 12 + df['month_num']

# Derived features
df['price_psm'] = df['price'] / df['area_sqm'].replace(0, np.nan)

# Clean strings
for c in ['market_segment','property_type','type_of_sale','tenure_type']:
    df[c] = df[c].astype(str).str.strip().str.upper()

# Drop rows with missing critical values
df = df.dropna(subset=['price','area_sqm','floor_mid','year','month_num','district'])
df = df[(df['price'] > 0) & (df['area_sqm'] > 0)]
df = df.reset_index(drop=True)
print(f'Clean dataset: {len(df):,} rows')

## 5  SORA Data (Google Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

SORA_PATH = '/content/drive/MyDrive/data/SORA_2017_2026.xlsx'
sora = pd.read_excel(SORA_PATH)[['SORA Publication Date','Compound SORA - 3 month']].copy()
sora.columns = ['date','sora_3m']
sora['date'] = pd.to_datetime(sora['date'])
sora['year']      = sora['date'].dt.year
sora['month_num'] = sora['date'].dt.month
sora_m = sora.groupby(['year','month_num'], as_index=False)['sora_3m'].mean()
sora_m = sora_m.rename(columns={'sora_3m':'sora'})

df = df.merge(sora_m, on=['year','month_num'], how='left')
df['sora'] = df['sora'].fillna(df['sora'].median())
print('SORA merged. Missing:', df['sora'].isna().sum())

## 6  Final Feature Set

In [ ]:
categorical_cols = ['market_segment','property_type','type_of_sale','tenure_type']
numerical_cols   = [
    'area_sqm','floor_mid','district','remaining_tenure_yrs',
    'no_of_units','year','quarter','time_idx','sora'
]
target_col = 'price'

# Fill any remaining NaNs in numerical cols with median
for c in numerical_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')
    df[c] = df[c].fillna(df[c].median())

FINAL = df[categorical_cols + numerical_cols + [target_col]].copy()
print(FINAL.shape)
FINAL.head(3)

## 7  Train / Test Split (time-based)
Train on transactions up to end-2023; test on 2024 onwards.

In [ ]:
train = FINAL[df['year'] < 2024]
test  = FINAL[df['year'] >= 2024]

X_train, y_train = train[categorical_cols + numerical_cols], train[target_col]
X_test,  y_test  = test[categorical_cols  + numerical_cols], test[target_col]

y_train_log = np.log(y_train)
y_test_log  = np.log(y_test)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

## 8  Base Model Pipelines

In [ ]:
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ('num', 'passthrough', numerical_cols)
])

def fresh_pre():
    return ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', 'passthrough', numerical_cols)
    ])

xgb_pipeline = Pipeline([('pre', fresh_pre()), ('model', XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    objective='reg:squarederror', n_jobs=-1
))])

lgb_pipeline = Pipeline([('pre', fresh_pre()), ('model', LGBMRegressor(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    n_jobs=-1, verbose=-1
))])

cat_pipeline = Pipeline([('pre', fresh_pre()), ('model', CatBoostRegressor(
    iterations=500, learning_rate=0.05, depth=6,
    random_seed=42, verbose=0
))])

## 9  Individual Model Evaluation

In [ ]:
def evaluate(name, pipeline, X_tr, X_te, y_tr, y_te):
    pipeline.fit(X_tr, np.log(y_tr))
    preds = np.exp(pipeline.predict(X_te))
    mae   = mean_absolute_error(y_te, preds)
    rmse  = np.sqrt(mean_squared_error(y_te, preds))
    r2    = r2_score(y_te, preds)
    mape  = mean_absolute_percentage_error(y_te, preds)
    print(f'{name:<20} MAE={mae:>12,.0f}  RMSE={rmse:>12,.0f}  R²={r2:.4f}  MAPE={mape*100:.2f}%')
    return preds

xgb_preds = evaluate('XGBoost',  xgb_pipeline, X_train, X_test, y_train, y_test)
lgb_preds = evaluate('LightGBM', lgb_pipeline, X_train, X_test, y_train, y_test)
cat_preds = evaluate('CatBoost', cat_pipeline, X_train, X_test, y_train, y_test)

## 10  Stacked Generalisation — HuberRegressor Metalayer
Base models generate **out-of-fold predictions** on the training set (5-fold CV).
These become the meta-features for the HuberRegressor, which is robust to outliers
in luxury/high-end transactions.

In [ ]:
print('Generating OOF predictions (5-fold CV)...')

xgb_oof_p = Pipeline([('pre', fresh_pre()), ('model', XGBRegressor(n_estimators=500,learning_rate=0.05,max_depth=6,subsample=0.8,colsample_bytree=0.8,random_state=42,objective='reg:squarederror',n_jobs=-1))])
lgb_oof_p = Pipeline([('pre', fresh_pre()), ('model', LGBMRegressor(n_estimators=500,learning_rate=0.05,num_leaves=63,subsample=0.8,colsample_bytree=0.8,random_state=42,n_jobs=-1,verbose=-1))])
cat_oof_p = Pipeline([('pre', fresh_pre()), ('model', CatBoostRegressor(iterations=500,learning_rate=0.05,depth=6,random_seed=42,verbose=0))])

xgb_oof = cross_val_predict(xgb_oof_p, X_train, y_train_log, cv=5, n_jobs=-1)
print('XGBoost OOF done')
lgb_oof = cross_val_predict(lgb_oof_p, X_train, y_train_log, cv=5, n_jobs=-1)
print('LightGBM OOF done')
cat_oof = cross_val_predict(cat_oof_p, X_train, y_train_log, cv=5)
print('CatBoost OOF done')

In [ ]:
# Train HuberRegressor metalayer on OOF predictions
meta_X_train = np.column_stack([xgb_oof, lgb_oof, cat_oof])
meta_model   = HuberRegressor(epsilon=1.35, max_iter=300)
meta_model.fit(meta_X_train, y_train_log)

print('Metalayer coefficients:')
for name, coef in zip(['XGBoost','LightGBM','CatBoost'], meta_model.coef_):
    print(f'  {name}: {coef:.4f}')
print(f'  Intercept: {meta_model.intercept_:.4f}')

In [ ]:
# Final stacked predictions on test set
meta_X_test   = np.column_stack([
    xgb_pipeline.predict(X_test),
    lgb_pipeline.predict(X_test),
    cat_pipeline.predict(X_test)
])
stacked_preds = np.exp(meta_model.predict(meta_X_test))

s_mae  = mean_absolute_error(y_test, stacked_preds)
s_rmse = np.sqrt(mean_squared_error(y_test, stacked_preds))
s_r2   = r2_score(y_test, stacked_preds)
s_mape = mean_absolute_percentage_error(y_test, stacked_preds)
print(f'Stacked (HuberMeta)   MAE={s_mae:>12,.0f}  RMSE={s_rmse:>12,.0f}  R²={s_r2:.4f}  MAPE={s_mape*100:.2f}%')

## 11  Evaluation Summary

In [ ]:
results = pd.DataFrame({
    'Model': ['XGBoost','LightGBM','CatBoost','Stacked (Huber)'],
    'MAE':   [mean_absolute_error(y_test, p) for p in [xgb_preds,lgb_preds,cat_preds]] + [s_mae],
    'RMSE':  [np.sqrt(mean_squared_error(y_test, p)) for p in [xgb_preds,lgb_preds,cat_preds]] + [s_rmse],
    'R2':    [r2_score(y_test, p) for p in [xgb_preds,lgb_preds,cat_preds]] + [s_r2],
    'MAPE%': [mean_absolute_percentage_error(y_test, p)*100 for p in [xgb_preds,lgb_preds,cat_preds]] + [s_mape*100]
})
results = results.sort_values('MAE').reset_index(drop=True)
print(results.to_string(index=False))

## 12  Sample Predictions

In [ ]:
sample = X_test.iloc[:10].copy()
actual = y_test.iloc[:10].values

s_xgb = np.exp(xgb_pipeline.predict(sample))
s_lgb = np.exp(lgb_pipeline.predict(sample))
s_cat = np.exp(cat_pipeline.predict(sample))
s_stk = np.exp(meta_model.predict(np.column_stack([
    xgb_pipeline.predict(sample),
    lgb_pipeline.predict(sample),
    cat_pipeline.predict(sample)
])))

out = pd.DataFrame({
    'Actual ($)':        actual,
    'XGBoost ($)':       s_xgb,
    'LightGBM ($)':      s_lgb,
    'CatBoost ($)':      s_cat,
    'Stacked-Huber ($)': s_stk,
})
out['Stacked Error %'] = (abs(out['Actual ($)'] - out['Stacked-Huber ($)']) / out['Actual ($)'] * 100).map('{:.2f}%'.format)
for c in ['Actual ($)','XGBoost ($)','LightGBM ($)','CatBoost ($)','Stacked-Huber ($)']:
    out[c] = out[c].map('${:,.0f}'.format)
out

## 13  Feature Importance (XGBoost)
Helps understand which features drive private property prices most.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Get feature names after preprocessing
xgb_fitted_pre = xgb_pipeline.named_steps['pre']
cat_names  = xgb_fitted_pre.named_transformers_['cat'].get_feature_names_out(categorical_cols).tolist()
feat_names = cat_names + numerical_cols

importances = xgb_pipeline.named_steps['model'].feature_importances_
fi = pd.Series(importances, index=feat_names).sort_values(ascending=False)[:20]

fig, ax = plt.subplots(figsize=(8,6))
fi.plot(kind='barh', ax=ax)
ax.set_title('Top 20 Feature Importances (XGBoost)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance_private.png', dpi=120)
plt.show()
print('Saved to feature_importance_private.png')

## 14  Save Model for Render Deployment
Download `private_model.pkl` from Colab and place it in `backend/` before deploying.

In [ ]:
import pickle

# Ensure base models are fitted on full training data
xgb_pipeline.fit(X_train, y_train_log)
lgb_pipeline.fit(X_train, y_train_log)
cat_pipeline.fit(X_train, y_train_log)

private_bundle = {
    'xgb': xgb_pipeline,
    'lgb': lgb_pipeline,
    'cat': cat_pipeline,
    'meta': meta_model,
    'categorical_cols': categorical_cols,
    'numerical_cols':   numerical_cols,
    'train_year_min':   int(df['year'].min()),
    'model_type':       'private_stacked_v1',
}

PKL_PATH = 'private_model.pkl'
with open(PKL_PATH, 'wb') as f:
    pickle.dump(private_bundle, f)
print(f'Saved → {PKL_PATH}')

try:
    from google.colab import files
    files.download(PKL_PATH)
except ImportError:
    print('Not in Colab — file saved locally.')